# Variant descriptive statistics

# 00 get the SNP and indels counts
```
module load VCFtools/0.1.16-GCC-10.3.0
current_date=$(date "+%Y%m%d")
for file in /ASA/analysis/WGS_results_cram/ASA_*/ASA_*/filtered_snp_indel/*vcf.gz
do
    name=${file#/ASA/analysis/WGS_results_cram/ASA_*/}
    name=${name%/filtered_snp_indel/merge_output_PASS.vcf.gz}
    printf $name
    printf " "
    less $file | vcf-annotate --fill-type | grep -oP "TYPE=\w+" | sort -r | uniq -c | grep -Eo "\b[0-9]+\b" | xargs
done | tee ${current_date}_snp_indel.rerun.txt
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as ticker


import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42

## 01 load variants Data V1

In [ ]:
variants = "/wgs/snp_indel.rerun.txt" # change path as necessary
vars = pd.read_csv(variants, sep=" ", header=None)
# add column names
vars.columns = ["sample", "SNP", "INS", "DEL"]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as ticker

# --- First panel: SNP, INS ≤ 50, DEL ≤ 50 ---
vars_melted = vars.melt(id_vars=["sample"], value_vars=["SNP", "INS", "DEL"], var_name="variant_type", value_name="count")
svtype_label_map1 = {
    'DEL': 'DEL ≤ 50',
    'INS': 'INS ≤ 50',
    'SNP': 'SNP'
}
vars_melted = vars_melted.copy()
vars_melted['variant_type'] = vars_melted['variant_type'].map(svtype_label_map1)

# --- Second and third panels: DEL ≥ 50, INS ≥ 50, DUP, BND, INV ---

svs = pd.read_csv('/ASA/analysis/WGS_pop_variants/SV/2026_SV/subset/sv.number_of_vars.txt', sep = ' ', header=None) # change as appropriate
#add headers BND, DEL, DUP, INS, INV
svs.columns = ['sample_id','BND', 'DEL', 'DUP', 'INS', 'INV']

melted = svs.melt(id_vars=['sample_id'], var_name='SV Type', value_name='Count')
filtered1 = melted[melted['SV Type'].isin(['DEL', 'INS'])]
filtered2 = melted[melted['SV Type'].isin(['DUP', 'BND', 'INV'])]

svtype_label_map2 = {
    'DEL': 'DEL > 50',
    'INS': 'INS > 50'
}
filtered1 = filtered1.copy()
filtered1['SV Type'] = filtered1['SV Type'].map(svtype_label_map2)


sns.set_style("ticks")
sns.set_context("talk", font_scale=1.6)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

# Panel 1
sns.boxplot(
    data=vars_melted, x="variant_type", y="count",
    showfliers=False, color="#F7A5A5", ax=axes[0], width=0.5
)
sns.stripplot(
    data=vars_melted, x="variant_type", y="count",
    color="black", jitter=0.3, alpha=0.3, ax=axes[0], size=5
)
axes[0].set_xlabel("")
axes[0].set_ylabel("Count", fontsize=14)
axes[0].set_ylim(bottom=0)

# Panel 2
sns.boxplot(
    data=filtered1, x="SV Type", y="Count",
    color="#F7A5A5", fliersize=0, linewidth=1.5, ax=axes[1], width=0.5
)
sns.stripplot(
    data=filtered1, x="SV Type", y="Count",
    jitter=0.3, color="black", alpha=0.3, ax=axes[1], size=5
)
axes[1].set_xlabel("")
axes[1].set_ylabel("Count", fontsize=14)

# Panel 3
sns.boxplot(
    data=filtered2, x="SV Type", y="Count",
    color="#F7A5A5", fliersize=0, linewidth=1.5, ax=axes[2], width=0.5
)
sns.stripplot(
    data=filtered2, x="SV Type", y="Count",
    jitter=0.3, color="black", alpha=0.3, ax=axes[2], size=5
)
axes[2].set_yscale("log")
axes[2].set_xlabel("")
axes[2].set_ylabel("Count (log)", fontsize=14)

def sci_notation(x, pos):
    if x == 0:
        return "0"
    exponent = int(np.floor(np.log10(abs(x))))
    coeff = x / 10**exponent
    if abs(coeff - 1) < 1e-2:
        return r"$10^{{{}}}$".format(exponent)
    return r"${:.1f} \times 10^{{{}}}$".format(coeff, exponent)

for ax in axes:
    legend = ax.get_legend()
    if legend:
        legend.set_frame_on(False)

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    ax.yaxis.set_major_formatter(ticker.FuncFormatter(sci_notation))

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=6,
        width=1,
        labelsize=13,
        bottom=True,
        left=True
    )

plt.tight_layout()
plt.savefig("figures/all_var_counts_boxplot.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# get descriptive statistics, but don't use e+06
stats = vars_melted.groupby("variant_type")["count"].describe()

# remove scientific notation
stats = stats.applymap(lambda x: f"{x:.0f}" if isinstance(x, (int, float)) else x)

stats

In [ ]:
# add metadata
metadata = "/ASA/analysis/donor_statistics/ega_metadata/metadata_run_expt_gut_incl.csv" # change path as appropriate
metadata = pd.read_csv(metadata)


metadata = metadata.rename(columns={'sample_name': 'sample'})
vars_combined_meta = vars.merge(metadata, left_on='sample', right_on='sample', how='left')
vars_combined_meta



In [ ]:
# plot vars_combined_meta INS and DELS with model as hue
vars_melted = vars_combined_meta.melt(id_vars=["model", "sample"], value_vars=["SNP", "INS", "DEL"], var_name="variant_type", value_name="count")      


vars_melted_indels = vars_melted[vars_melted["variant_type"].isin(["INS", "DEL"])]
plt.figure(figsize=(15, 6))
sns.boxplot(data=vars_melted_indels, x="variant_type", y="count", hue="model", showfliers=False, palette="Set2")
sns.stripplot(data=vars_melted_indels, x="variant_type", y="count", hue="model", dodge=True, color="black", jitter=True, alpha=0.6, legend=False)
plt.title("Indel Counts by ONT model", fontsize=16)
plt.xlabel("Variant Type", fontsize=14)
plt.ylabel("Count", fontsize=14)
plt.yscale("log")
plt.legend(title="ONT model", bbox_to_anchor=(1.05, 1), loc='upper left', frameon = False)
plt.tight_layout()
plt.show()

In [ ]:
font_size = 25

vars_melted = vars_combined_meta.melt(id_vars=["model", "sample"], value_vars=["SNP", "INS", "DEL"], var_name="variant_type", value_name="count")      
vars_melted_indels = vars_melted[vars_melted["variant_type"].isin(["INS", "DEL"])]

variant_types = vars_melted["variant_type"].unique()
all_models = vars_melted['model'].unique()
ordered_models = sorted(all_models, key=lambda x: (not ("r9.4.1" in x), x))

fig, axes = plt.subplots(len(variant_types), 1, figsize=(15, 6 * len(variant_types)), sharex=True)

for i, var_type in enumerate(variant_types):
    ax = axes[i]
    vars_melted_indels = vars_melted[vars_melted["variant_type"] == var_type].copy()
    vars_melted_indels['model'] = pd.Categorical(vars_melted_indels['model'], categories=ordered_models, ordered=True)

    sns.boxplot(
        data=vars_melted_indels,
        x="model",
        y="count",
        showfliers=False,
        palette="Set2",
        ax=ax
    )
    sns.stripplot(
        data=vars_melted_indels,
        x="model",
        y="count",
        dodge=True,
        color="black",
        jitter=True,
        alpha=0.6,
        legend=False,
        ax=ax
    )
    ax.set_title(f"{var_type} Counts", fontsize=font_size)
    ax.set_ylabel("Count", fontsize=font_size)
    ax.set_yscale("log")
    ax.grid(True, axis='y', linestyle='--', alpha=0.5)
    ax.tick_params(axis='y', labelsize=font_size)
    ax.tick_params(axis='x', labelsize=font_size)

axes[-1].set_xlabel("ONT Model", fontsize=font_size)

# Set x-tick label font size explicitly for all axes
for ax in axes:
    for label in ax.get_xticklabels():
        label.set_fontsize(font_size)

plt.xticks(rotation=45, ha='right', fontsize=font_size)
plt.tight_layout()
plt.savefig('figures/small_var_counts_boxplot_model.pdf', bbox_inches='tight')
plt.show()

In [ ]:
pip list